<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

#| export: true
Detection is PPLCNetV4, a RepLKPAN neck, and the inference DB head. Recognition uses the same backbone with height-only downsampling, a LightSVTR neck, and CTC. The official ONNX checkpoints supply the weights. The notebooks in `nbs/` are the source; `nbdev-export` writes this package.

## Install

The graph runtime is [Modular MAX](https://docs.modular.com/max/), installed here with [pixi](https://pixi.sh). Checkpoints are local and are not part of the package.

```sh
pixi install
pixi run download
```

## Read a page

`read_page` returns each line as text, detector confidence, and an axis-aligned box in the original image.

In [ ]:
#| eval: false
from pathlib import Path
from solid_lens.core import repo_root
from solid_lens.model import compile_detector, compile_recognizer
from solid_lens.ocr import detection_hw, load_charset, load_image, read_page

root = repo_root()
image = load_image(Path("page.png"))
det_h, det_w = detection_hw(*image.shape[:2])
detector = compile_detector(
    height=det_h, width=det_w,
    cache_dir=root/"models"/f"native_medium_det_{det_h}x{det_w}.mef")
recognizer = compile_recognizer(cache_dir=root/"models"/"native_medium_rec_48x320.mef")
charset = load_charset(root/"models"/"ppocrv6_dict.txt", add_space=True)
read_page(image, detector, recognizer, charset)

The same pipeline is a command, and a JSON service on port 5001:

```sh
pixi run ocr page.png
pixi run serve
```

`POST` one or more files to `/ocr` as `images` or `image`. Each result has the file name and a list of fragments: `text`, `norm`, `score`, and `box`.

## Develop

Edit the notebooks. The modules under `solid_lens/` are generated, and each one names the notebook to change.

```sh
nbdev-export
nbdev-test
```

`nbdev-test` runs the lessons that do not need MAX. `nbdev-test --flags notest` also executes the cells that build graphs, which needs the pixi environment.

The reading order follows a forward pass. `config` is the architecture. `onnx` reads a checkpoint. The layer notebooks are the blocks `max.nn` does not ship. `detector` and `recognizer` stack them, `weights` binds the checkpoint, `model` compiles, and `ocr` is the page pipeline.